# M2: Frozen MiniLM Metadata Representations

Final semantic metadata path: title, categories, features and description; excluded preliminary TF-IDF and store-field experiments.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

DATA_DIR = Path('.')
metadata = pd.read_parquet(DATA_DIR / 'product_metadata.parquet', columns=['parent_asin', 'title', 'categories', 'features', 'description']).reset_index(drop=True)
print('Catalog:', metadata.shape)


In [ ]:
import numpy as np
import pandas as pd


def clean_field(value):
    if value is None:
        return ""

    if isinstance(value, (list, tuple, np.ndarray)):
        return " ".join(
            str(x) for x in value
            if x is not None
        )

    try:
        if pd.isna(value):
            return ""
    except:
        pass

    return str(value)


metadata = metadata.copy()

for col in [
    "title",
    "categories",
    "features",
    "description"
]:
    metadata[col + "_text"] = (
        metadata[col]
        .apply(clean_field)
    )


metadata["content_text"] = (
    "Title: " + metadata["title_text"]
    + ". Categories: " + metadata["categories_text"]
    + ". Features: " + metadata["features_text"]
    + ". Description: " + metadata["description_text"]
)

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Device:", encoder.device)
print("Embedding dimension:", encoder.get_sentence_embedding_dimension())
print("Max sequence length:", encoder.max_seq_length)

In [ ]:
item_embeddings = encoder.encode(
    metadata["content_text"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(item_embeddings.shape)
print(item_embeddings.dtype)

In [ ]:
metadata = metadata.reset_index(drop=True)

print(
    "Metadata rows:",
    len(metadata)
)

print(
    "Unique ASINs:",
    metadata["parent_asin"].nunique()
)

assert metadata["parent_asin"].is_unique

In [ ]:
np.save(
    DATA_DIR / "metadata_minilm_embeddings.npy",
    item_embeddings
)

metadata[["parent_asin"]].to_parquet(
    DATA_DIR / "metadata_embedding_index.parquet",
    index=False
)